# Image Generation Setup

Thesis: *Experimental Setup*, section *Image Generation and Output-Level Classification*, subsection *Image Generation Setup*.

**Requirements**
- Environment: `easy-language` (`environment-easy-language.yml`)
- Packages: transformers, tokenizers, huggingface_hub
- Data: `data/corpus/sd_prompts.csv`, `data/corpus/ls_prompts.csv` (in this repository)
- Network: the tokenizer of `black-forest-labs/FLUX.2-klein-4B` at a pinned revision, a few MB, public
- GPU: not needed

**Writes:** `exports/tokenization_check.csv`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "corpus").is_dir())
os.chdir(ROOT)
for folder in ("corpus", "generation"):
    sys.path.insert(0, str(ROOT / "src" / folder))


from transformers import Qwen2TokenizerFast

import corpus_spec as spec
from token_alignment import FLUX_REVISION, find_span_token_indices, templated_prompt_alignment
import pandas as pd
from IPython.display import display

## Tokenization of the realised prompts

Whether the tokenizer breaks the gender forms in the realised prompts. The typographical form keeps the tokens of the generic masculine and adds three for the markers, the neutral terms fragment whether attested or not, and nothing is truncated.

In [2]:
MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
MAX_SEQUENCE_LENGTH = 256  # matches the actual generation call, not the pipeline's default 512

PROMPT_FILES = {"sd": Path("data/corpus/sd_prompts.csv"), "ls": Path("data/corpus/ls_prompts.csv")}
OUT_PATH = Path("exports/tokenization_check.csv")
OUT_FIELDS = [
    "profession", "condition", "variety", "scene_label", "mention_index",
    "actor_phrase", "n_tokens", "token_pieces", "template_offset", "truncated",
]

NEUTRAL_TERM_ATTESTATION = {
    "mechaniker":      ("Kfz-Fachkraft",      "marginal (77 hits, ~0.0014/M)"),
    "elektriker":      ("Elektrofachkraft",   "attested (0.21/M)"),
    "reinigungskraft": ("Reinigungskraft",    "attested (1.23/M, Duden)"),
    "florist":         ("Blumenfachkraft",    "unattested (0 hits, no Duden entry)"),
    "baecker":         ("Bäckereifachkraft",  "unattested (26 hits, ~0.0005/M, no Duden entry)"),
    "fotograf":        ("Fotofachkraft",      "unattested (7 hits, no Duden entry)"),
}


def load_unique_rows() -> list[dict]:
    """One row per realised (profession, condition, variety, scene_label) - seeds don't change
    prompt_text (corpus_spec.py instantiates the same prompt across all of spec.SEEDS), so
    seed==first seed is enough to cover every distinct realised prompt without redundant re-
    tokenization.
    """
    rows = []
    for path in PROMPT_FILES.values():
        prompts = pd.read_csv(path, dtype=str, keep_default_na=False)
        rows.extend(prompts[prompts["seed"] == str(spec.SEEDS[0])].to_dict("records"))
    return rows

In [3]:
tokenizer = Qwen2TokenizerFast.from_pretrained(MODEL_ID, subfolder="tokenizer", revision=FLUX_REVISION)

In [4]:
out_rows = []
canonical_pieces: dict[tuple, list[str]] = {}
variance_notes = []

for row in load_unique_rows():
    prof, cond, variety, scene = row["profession"], row["condition"], row["variety"], row["scene_label"]
    prompt = row["prompt_text"]
    if cond not in spec.ACTORS[prof]:
        continue
    actor = spec.ACTORS[prof][cond][0]

    offsets, template_offset, _ = templated_prompt_alignment(tokenizer, prompt, MAX_SEQUENCE_LENGTH)
    bare_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]

    needles = [actor]
    if variety == "ls":
        definite = spec.definite_actor(actor)
        if definite in prompt:
            needles.append(definite)  # repeated mention (rule A3) - only present in P1/P3/P4 frames

    search_from = 0
    for mention_idx, needle in enumerate(needles, start=1):
        idxs, char_end = find_span_token_indices(offsets, template_offset, prompt, needle, search_from)
        search_from = char_end
        pieces = tokenizer.convert_ids_to_tokens([bare_ids[i - template_offset] for i in idxs])
        truncated = any(i >= MAX_SEQUENCE_LENGTH for i in idxs)

        out_rows.append({
            "profession": prof, "condition": cond, "variety": variety, "scene_label": scene,
            "mention_index": mention_idx, "actor_phrase": needle,
            "n_tokens": len(idxs), "token_pieces": "|".join(pieces),
            "template_offset": template_offset, "truncated": truncated,
        })

        key = (prof, cond, variety, mention_idx)
        if key in canonical_pieces and canonical_pieces[key] != pieces:
            variance_notes.append(f"{key} scene={scene}: {canonical_pieces[key]} vs {pieces}")
        canonical_pieces.setdefault(key, pieces)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(out_rows, columns=OUT_FIELDS).to_csv(OUT_PATH, index=False, lineterminator="\r\n")
print(f"wrote {len(out_rows)} rows -> {OUT_PATH}")

wrote 636 rows -> exports/tokenization_check.csv


In [5]:
truncated_rows = [r for r in out_rows if r["truncated"]]
print(f"truncated actor-phrase tokens (>= {MAX_SEQUENCE_LENGTH}): {len(truncated_rows)}")

if variance_notes:
    print(f"\nWARNING: {len(variance_notes)} scene-dependent tokenization mismatches (unexpected):")
    for note in variance_notes:
        print(" ", note)
else:
    print("\nNo scene-dependent variance: every (profession, gender form, language form, mention) "
          "tokenizes identically across all scenes it appears in.")

truncated actor-phrase tokens (>= 256): 0

No scene-dependent variance: every (profession, gender form, language form, mention) tokenizes identically across all scenes it appears in.


In [6]:
print("Typographical form against generic masculine, first mention, GSL")
typographical = []
for prof in spec.PROFESSIONS:
    gm = canonical_pieces.get((prof, "generic_masc", "sd", 1))
    star = canonical_pieces.get((prof, "typographical", "sd", 1))
    if gm is None or star is None:
        continue
    typographical.append({"profession": prof, "generic masculine": len(gm),
                          "typographical form": len(star), "extra tokens": len(star) - len(gm),
                          "generic masculine pieces": " | ".join(gm),
                          "typographical form pieces": " | ".join(star)})
display(pd.DataFrame(typographical).set_index("profession"))

print("Neutral terms: tokens against corpus attestation, first mention, GSL")
neutral = []
for prof, (term, status) in NEUTRAL_TERM_ATTESTATION.items():
    pieces = canonical_pieces.get((prof, "neutral_term", "sd", 1))
    if pieces is None:
        continue
    neutral.append({"profession": prof, "neutral term": term, "attestation": status,
                    "tokens": len(pieces), "pieces": " | ".join(pieces)})
display(pd.DataFrame(neutral).set_index("profession"))

Typographical form against generic masculine, first mention, GSL


,generic masculine,typographical form,extra tokens,generic masculine pieces,typographical form pieces
profession,,,,,
mechaniker,8,11,3,Ein | ĠK | f | z | -M | ech | an | iker,Ein | *e | ĠK | f | z | -M | ech | an | iker |...
elektriker,5,8,3,Ein | ĠEle | k | tr | iker,Ein | *e | ĠEle | k | tr | iker | * | in
reinigungskraft,6,9,3,Ein | ĠRa | ump | f | le | ger,Ein | *e | ĠRa | ump | f | le | ger | * | in
florist,3,6,3,Ein | ĠFlor | ist,Ein | *e | ĠFlor | ist | * | in
baecker,4,7,3,Ein | ĠB | Ã¤ | cker,Ein | *e | ĠB | Ã¤ | cker | * | in
fotograf,3,6,3,Ein | ĠFot | ograf,Ein | *e | ĠFot | ograf | * | in


Neutral terms: tokens against corpus attestation, first mention, GSL


,neutral term,attestation,tokens,pieces
profession,,,,
mechaniker,Kfz-Fachkraft,"marginal (77 hits, ~0.0014/M)",8,Eine | ĠK | f | z | -F | ach | k | raft
elektriker,Elektrofachkraft,attested (0.21/M),8,Eine | ĠEle | kt | ro | f | ach | k | raft
reinigungskraft,Reinigungskraft,"attested (1.23/M, Duden)",5,Eine | ĠRein | igung | sk | raft
florist,Blumenfachkraft,"unattested (0 hits, no Duden entry)",7,Eine | ĠBl | umen | f | ach | k | raft
baecker,Bäckereifachkraft,"unattested (26 hits, ~0.0005/M, no Duden entry)",9,Eine | ĠB | Ã¤ | ck | ere | if | ach | k | raft
fotograf,Fotofachkraft,"unattested (7 hits, no Duden entry)",6,Eine | ĠFot | of | ach | k | raft
